In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import torch
import torchvision.transforms as T
import PIL.Image as Image

/home/mattia/miniconda3/envs/keypoint_factory/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
image_path = "DJI_0013_frame_000023.jpg"
img = Image.open(image_path).convert("RGB")

img_tensor = T.ToTensor()(img).cuda()[None]
print(img_tensor.shape)

In [ ]:
from extractors.canny import CannyEdgeDetector

edge_extractor = CannyEdgeDetector(device="cuda", **{"sigma": 3.0})
edges = edge_extractor(img_tensor)
edges.shape, edges.device

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(~(edges.squeeze().cpu().bool()), cmap="gray")
plt.show()

In [ ]:
import pycolmap 

rec = pycolmap.Reconstruction("sparse/0")

In [ ]:
import h5py
import torch
import pycolmap
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

def load_reconstruction(recon_path):
    """Load COLMAP reconstruction."""
    recon = pycolmap.Reconstruction(recon_path)
    cams = recon.cameras
    imgs = recon.images
    id_to_name = {img.image_id: img.name for img in imgs.values()}
    return recon, cams, imgs, id_to_name


def estimate_depth_limits(depth_dir, image_name, default_near=0.1, default_far=5.0):
    """
    Estimate z_near and z_far from per-image depth map (.npy or .exr).
    If not found, return defaults.
    """
    depth_path = Path(depth_dir) / (image_name.split('.')[0] + '.h5')
    if not depth_path.exists():
        print(f"Depth map not found for {depth_path}, using defaults: near={default_near}, far={default_far}")
        return default_near, default_far

    # load h5 file
    depth = h5py.File(depth_path, "r")["depth"][()]
    valid = depth[np.isfinite(depth)]

    if valid.size < 10:
        return default_near, default_far
    
    near = np.percentile(valid, 5)
    far = np.percentile(valid, 95)

    return float(near), float(far)


def compute_frustum_corners(K, width, height, z_near, z_far, R, t, device):
    """Compute 8 frustum corners in world coordinates."""
    corners_px = torch.tensor([
        [0, 0, 1],
        [width, 0, 1],
        [width, height, 1],
        [0, height, 1]
    ], dtype=torch.float32, device=device)

    invK = torch.inverse(K)
    near_pts = (invK @ corners_px.T).T * z_near
    far_pts = (invK @ corners_px.T).T * z_far
    pts_cam = torch.cat([near_pts, far_pts], dim=0)
    # Xw = R^T (Xc - t)
    Xw = (R.T @ (pts_cam.T - t.reshape(3, 1))).T
    return Xw


def aabb_from_points(points):
    """Compute axis-aligned bounding box."""
    return points.min(dim=0).values, points.max(dim=0).values


def aabb_overlap(a_min, a_max, b_min, b_max):
    """Check if AABBs intersect."""
    return torch.all(a_min <= b_max) and torch.all(b_min <= a_max)


def build_view_graph_from_frustums(
    recon_path,
    depth_dir=None,
    z_near_default=0.1,
    z_far_default=5.0,
    max_view_angle_deg=75.0,
    distance_factor=2,
    verbose=True,
):
    """
    Compute view-graph image pairs by frustum intersection, 
    with tighter geometric filtering.
    Args:
        recon_path: path to COLMAP reconstruction folder
        depth_dir: optional path to per-image depth maps (for better near/far estimation)
        device: torch device
        z_near_default: default near plane distance
        z_far_default: default far plane distance
        max_view_angle_deg: maximum allowed view-direction angle difference between cameras. (To reduce pairs: lower the value (e.g., 20°)
        distance_factor: maximum allowed distance between camera centers as a factor of scene size (To reduce pairs: set to 1.0-1.5, to increase pairs: set to 3.0-4.0.)
    """

    recon, cams, imgs, id_to_name = load_reconstruction(recon_path)
    device = "cpu"

    frustums = {}
    aabbs = {}
    centers = {}
    directions = {}

    if verbose is True:
        print("Building camera frustums and computing metadata...")
        bar = tqdm(imgs.values())
    else:
        bar = imgs.values()

    for img in bar:
        cam = cams[img.camera_id]
        K = torch.tensor(cam.calibration_matrix(), dtype=torch.float32, device=device)
        R = torch.tensor(img.cam_from_world.rotation.matrix(), dtype=torch.float32, device=device)
        t = torch.tensor(img.cam_from_world.translation, dtype=torch.float32, device=device)

        if depth_dir:
            z_near, z_far = estimate_depth_limits(depth_dir, img.name)
        else:
            z_near, z_far = z_near_default, z_far_default

        # shrink far plane slightly (avoid wide skinny cones)
        z_far *= 0.9

        corners = compute_frustum_corners(K, cam.width, cam.height, z_near, z_far, R, t, device)
        aabbs[img.image_id] = aabb_from_points(corners)
        frustums[img.image_id] = corners

        # camera center in world = -R^T t
        c_world = -(R.T @ t)
        centers[img.image_id] = c_world
        # camera forward vector in world
        d_world = R.T @ torch.tensor([0., 0., 1.], device=device)
        directions[img.image_id] = d_world / torch.norm(d_world)

    ids = list(imgs.keys())
    pairs = []

    cos_angle_thresh = torch.cos(torch.deg2rad(torch.tensor(max_view_angle_deg, device=device)))

    if verbose:
        print(f"\nChecking {len(ids)} cameras for tight frustum overlaps...")

    for i_idx, i in enumerate(ids):
        a_min, a_max = aabbs[i]
        ci, di = centers[i], directions[i]
        for j in ids[i_idx + 1:]:
            b_min, b_max = aabbs[j]
            cj, dj = centers[j], directions[j]

            # Step 1: AABB intersection (coarse)
            if not aabb_overlap(a_min, a_max, b_min, b_max):
                continue

            # Step 2: view direction consistency
            cos_angle = torch.dot(di, dj)
            if cos_angle < cos_angle_thresh:
                continue  # too divergent (e.g., opposite sides)

            # Step 3: distance filter
            dist = torch.norm(ci - cj)
            scene_scale = torch.norm(a_max - a_min)
            if dist > distance_factor * scene_scale:
                continue

            pairs.append([i, j])

    if verbose:
        print(f"\nFound {len(pairs):,} tight overlapping pairs.")
    return pairs, id_to_name


def save_pairs_to_file(pairs, id_to_name, output_file="pairs.txt"):
    """Save view-graph pairs to text file (COLMAP-style)."""
    out_pairs = []
    for i, j in pairs:
        sorted_ij = sorted([id_to_name[i], id_to_name[j]])
        out_pairs.append([sorted_ij[0], sorted_ij[1]])
    
    if output_file is None:
        with open("pairs_debug.txt", "w") as f:
            for i, j in out_pairs:
                f.write(f"{i} {j}\n")

    # sort pairs by first image name and then second image name
    out_pairs = sorted(out_pairs, key=lambda x: (x[0], x[1]))

    return out_pairs


In [ ]:
# Define paths
recon_path = "/home/mattia/Desktop/datasets/mydataset/data/udine_castello_fagagna/colmap/sparse/0"
depth_dir = "/home/mattia/Desktop/datasets/mydataset/data/udine_castello_fagagna/depth"

# Run view-graph building
pairs, id_to_name = build_view_graph_from_frustums(
    recon_path,
    depth_dir=None, # not changing much, so we skip depth maps for speed
    verbose=True,
)

# Optionally save
pairs = save_pairs_to_file(pairs, id_to_name, None) #"sparse/pairs.txt")

pairs[:5]

In [ ]:
# # load pairs
# with open("sparse/pairs.txt", "r") as f:
#     pairs = [line.strip().split() for line in f.readlines()]    

for n in [15, 30, 50]:
    # load viewgraph_15.txt
    with open(f"/home/mattia/Desktop/datasets/mydataset/data/udine_castello_fagagna/colmap/viewgraph_{n}.txt", "r") as f:
        viewgraph = [line.strip().split()[:2] for line in f.readlines()]

    # Convert to tuple-based set once
    viewgraph_set = {tuple(p) for p in viewgraph}
    overlap_pairs = {(a, b) for (a, b) in pairs if (a, b) in viewgraph_set or (b, a) in viewgraph_set}
    print(f"Viewgraph pairs ({n}): {len(viewgraph):,}, Found pairs: {len(pairs):,}, Correct pairs found: {len(overlap_pairs):,} ({len(overlap_pairs)/len(viewgraph)*100:.2f}%)")


In [ ]:
import os

res = {}
for scene in tqdm(sorted(os.listdir("/home/mattia/Desktop/datasets/mydataset/data"))):

    # Define paths
    recon_path = f"/home/mattia/Desktop/datasets/mydataset/data/{scene}/colmap/sparse/0"

    # Run view-graph building
    pairs, id_to_name = build_view_graph_from_frustums(
        recon_path,
        depth_dir=None,
        verbose=False,
    )

    pairs = save_pairs_to_file(pairs, id_to_name, None)  #"sparse/pairs.txt")

    # load viewgraph_30.txt
    with open(f"/home/mattia/Desktop/datasets/mydataset/data/{scene}/colmap/viewgraph_30.txt", "r") as f:
        viewgraph = [line.strip().split()[:2] for line in f.readlines()]

    # Convert to tuple-based set once
    viewgraph_set = {tuple(p) for p in viewgraph}
    overlap_pairs = {(a, b) for (a, b) in pairs if (a, b) in viewgraph_set or (b, a) in viewgraph_set}
    # print(f"Viewgraph pairs ({n}): {len(viewgraph):,}, Found pairs: {len(pairs):,}, Correct pairs found: {len(overlap_pairs):,} ({len(overlap_pairs)/len(viewgraph)*100:.2f}%)")

    res[scene] = {
        "viewgraph_pairs": len(viewgraph),
        "found_pairs": len(pairs),
        "overlap_pairs": len(overlap_pairs),
    }

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(res, orient="index")
df["correctness"] = df["overlap_pairs"] / df["viewgraph_pairs"]
df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 6))
plt.bar(df.index, df["correctness"])
# plot median and avg lines
plt.axhline(y=df["correctness"].mean(), color='r', linestyle='--', label='Average')
plt.axhline(y=df["correctness"].median(), color='g', linestyle='--', label='Median')
plt.legend()
plt.xticks(rotation=90)
plt.ylabel("Correctness")
plt.show()